# Importing Libraries

In [ ]:
!pip install --upgrade imutils

In [ ]:
from keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from keras.layers.convolutional import Conv2D, MaxPooling2D
from keras.layers.core import Activation, Flatten, Flatten, Dropout, Dense
from keras import backend as K
from tensorflow.keras.utils import img_to_array, load_img
from keras.preprocessing.image import ImageDataGenerator
from keras.optimizers import Adam
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os
from keras import preprocessing
import random

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sea
from tqdm.notebook import tqdm
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn.functional as F
import torchmetrics
from torchmetrics import Metric
import torchvision
from torchvision import models
import cv2 as op

plt.style.use('seaborn')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

np.__version__, device

In [ ]:
class SmallerVGGNet:
    @staticmethod
    def build(width, height, depth, classes, finalAct="softmax"):
        model = Sequential()
        inputShape = (height, width, depth)

        # CONV => RELU => POOL
        model.add(Conv2D(32, (3, 3), padding="same",
                         input_shape=inputShape))
        model.add(Activation("relu"))
        model.add(BatchNormalization())
        model.add(MaxPooling2D(pool_size=(3, 3)))
        model.add(Dropout(0.25))

        # CONV => RELU => CONV => RELU => POOL
        model.add(Conv2D(64, (3, 3), padding="same"))
        model.add(Activation("relu"))
        model.add(BatchNormalization())
        model.add(Conv2D(64, (3, 3), padding="same"))
        model.add(Activation("relu"))
        model.add(BatchNormalization())
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(Dropout(0.25))

        # CONV => RELU => CONV => RELU => POOL
        model.add(Conv2D(128, (3, 3), padding="same"))
        model.add(Activation("relu"))
        model.add(BatchNormalization())
        model.add(Conv2D(128, (3, 3), padding="same"))
        model.add(Activation("relu"))
        model.add(BatchNormalization())
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(Dropout(0.25))

        # FC => RELU
        model.add(Flatten())
        model.add(Dense(1024))
        model.add(Activation("relu"))
        model.add(BatchNormalization())
        model.add(Dropout(0.5))

        # Output
        model.add(Dense(classes))
        model.add(Activation(finalAct))

        return model

# Loading the data

In [ ]:
TRAIN_PATH = '/kaggle/input/fresh-and-stale-classification/dataset/Train'
TEST_PATH = '/kaggle/input/fresh-and-stale-classification/dataset/Test'

def load_data(PATH):
    filenames, fruit, fresh = [], [], []
    
    for file in tqdm(os.listdir(PATH)):
        for img in os.listdir(os.path.join(PATH, file)):
            fresh.append(0 if file[0] == 'f' else 1)
            fruit.append(file[5:] if file[0] == 'f' else file[6: ])
            filenames.append(os.path.join(PATH, file, img))
            
    df = pd.DataFrame({
        'filename' : filenames,
        'fruit' : fruit,
        'fresh' : fresh
    })
    
    return df

df_train = load_data(TRAIN_PATH)
df_test = load_data(TEST_PATH)

df_train.shape, df_test.shape
    

## Data Preprocessing

In [ ]:
df_train.info()

In [ ]:
df_test.info()

In [ ]:
print(df_train['fruit'].value_counts())

In [ ]:
print(df_test['fruit'].value_counts())

In [ ]:
# threshold_counts = {'apples': 2000, 'banana': 2000,
#                     'oranges': 2000, 'tomato': 2000}

# for fruit, threshold in threshold_counts.items():
#     fruit_indices = df[df['fruit'] == fruit].index
#     if len(fruit_indices) > threshold:
#         rows_to_drop = fruit_indices[:len(fruit_indices) - threshold]
#         df = df.drop(rows_to_drop)

# print(df['fruit'].value_counts())

In [ ]:
# apples_indices = df[df['fruit'] == 'apples'].index
# banana_indices = df[df['fruit'] == 'banana'].index
# oranges_indices = df[df['fruit'] == 'oranges'].index
# tomato_indices = df[df['fruit'] == 'tomato'].index

# if len(apples_indices) > 2000:
#     rows_to_drop = apples_indices[:len(apples_indices) - 2000]
#     df = df.drop(rows_to_drop)

# if len(banana_indices) > 2000:
#     rows_to_drop = banana_indices[:len(banana_indices) - 2000]
#     df = df.drop(rows_to_drop)
    
# if len(oranges_indices) > 2000:
#     rows_to_drop = oranges_indices[:len(oranges_indices) - 2000]
#     df = df.drop(rows_to_drop)
    
# if len(tomato_indices) > 2000:
#     rows_to_drop = tomato_indices[:len(tomato_indices) - 2000]
#     df = df.drop(rows_to_drop)

# print(df['fruit'].value_counts())

In [ ]:
sea.countplot(x = 'fruit', data = df_train, hue = 'fresh')

In [ ]:
sea.countplot(x = 'fruit', data = df_test, hue = 'fresh')

## New balanced dataset

In [ ]:
from sklearn.preprocessing import LabelEncoder

LABEL_ALIASES = {'patato': 'potato', 'tamto': 'tomato'}
df_train['fruit'] = df_train['fruit'].replace(LABEL_ALIASES)
df_test['fruit'] = df_test['fruit'].replace(LABEL_ALIASES)

fruit_encoder = LabelEncoder()
df_train['fruit_label'] = fruit_encoder.fit_transform(df_train['fruit'])
df_train.shape


In [ ]:
# Reuse the training encoder so every numeric label has the same meaning.
unknown_test_labels = set(df_test['fruit']) - set(fruit_encoder.classes_)
if unknown_test_labels:
    raise ValueError(f'Unknown test labels: {sorted(unknown_test_labels)}')

df_test['fruit_label'] = fruit_encoder.transform(df_test['fruit'])
df_test.shape


In [ ]:
df_train.head()

In [ ]:
df_test.head()

In [ ]:
sea.countplot(x = 'fruit', data = df_train, hue = 'fresh')

In [ ]:
sea.countplot(x = 'fruit', data = df_test, hue = 'fresh')

In [ ]:
df_train['fruit']

In [ ]:
df_test['fruit']

## Split train,test and validate

In [ ]:
from sklearn.model_selection import train_test_split
df_train['fresh_fruit'] = df_train['fresh'].astype(str) + "_" + df_train['fruit_label'].astype(str)
df_train, df_val = train_test_split(df_train, test_size=0.2, stratify=df_train['fresh_fruit'])
df_train = df_train.drop('fresh_fruit', axis=1)
df_val=df_val.drop('fresh_fruit', axis=1)
df_train.shape, df_val.shape, df_test.shape

In [ ]:
sea.countplot(x = 'fruit_label', data = df_train, hue = 'fresh', palette = 'Blues_d')

In [ ]:
sea.countplot(x = 'fruit', data = df_val, hue = 'fresh', palette = 'Blues_d')

In [ ]:
def load_image(path):
    img = plt.imread(path)
    return img

counter = 0

plt.figure(figsize = (10, 30))

for i in range(9):
    for path in df_train[df_train['fruit_label'] == i].sample(n = 3)['filename']:
        plt.subplot(9, 3, counter + 1)
        img = load_image(path)
        plt.imshow(img)
        plt.axis('off')
        plt.title('Class:' + " " + fruit_encoder.inverse_transform([i]))
        counter += 1
        
plt.show()

# Building the dataset

In [ ]:

def image_transform(img, p = 0.5, training = True):    
    if training:
        img = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p = p),
            transforms.GaussianBlur(3, sigma=(0.1, 2.0)),
            transforms.RandomAdjustSharpness(3, p = p),
            transforms.Normalize(mean = 0, std = 1)
        ])(img)
    else:
        img = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((224, 224)),
            transforms.Normalize(mean = 0, std = 1)
        ])(img)

    return img

class FruitDataset:
    def __init__(self, df, training):
        self.df = df
        self.n_samples = len(self.df)
        self.training = training
        
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        img = plt.imread(self.df.iloc[idx][0])[:, :, :3]
        fresh = torch.tensor(self.df.iloc[idx][2])
        fruit = torch.tensor(self.df.iloc[idx][3])

        img = image_transform(img, p = 0.5, training = self.training)
            
        return img, fruit, fresh
    

In [ ]:
BATCH_SIZE = 64


train_dataset = FruitDataset(df_train, training = True)
val_dataset = FruitDataset(df_val, training = False)
test_dataset = FruitDataset(df_test, training = False)

train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE)


In [ ]:
a, b, c = next(iter(train_loader))
print(a.shape, b.shape, c.shape)
del(a)
del(b)
del(c)

# Train Model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from tqdm import tqdm


class ModelVGG16(nn.Module):
    def __init__(self):
        super().__init__()
        self.alpha = 0.7
        self.base = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

        for param in list(self.base.parameters())[:-15]:
            param.requires_grad = False

        self.base.classifier = nn.Sequential()
        self.block1 = nn.Sequential(
            nn.Linear(512 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
        )
        self.block2 = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 9),
        )
        self.block3 = nn.Sequential(
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 2),
        )
        self.loss_fxn = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam([
            {'params': self.base.parameters(), 'lr': 1e-5},
            {'params': self.block1.parameters(), 'lr': 3e-4},
            {'params': self.block2.parameters(), 'lr': 3e-4},
            {'params': self.block3.parameters(), 'lr': 3e-4},
        ])
        self.history = {
            'train_loss': [], 'val_loss': [],
            'train_acc_fruit': [], 'train_acc_fresh': [],
            'val_acc_fruit': [], 'val_acc_fresh': [],
        }

    def forward(self, x):
        x = torch.flatten(self.base.features(x), 1)
        x = self.block1(x)
        return self.block2(x), self.block3(x)

    def _run_epoch(self, loader, training):
        super().train(training)
        total_loss = 0.0
        fruit_correct = 0
        fresh_correct = 0
        sample_count = 0

        for x, fruit_target, fresh_target in tqdm(loader):
            x, fruit_target, fresh_target = [value.to(device) for value in (x, fruit_target, fresh_target)]
            self.optimizer.zero_grad()

            with torch.set_grad_enabled(training):
                fruit_logits, fresh_logits = self(x)
                fruit_loss = self.loss_fxn(fruit_logits, fruit_target)
                fresh_loss = self.loss_fxn(fresh_logits, fresh_target)
                loss = self.alpha * fruit_loss + (1 - self.alpha) * fresh_loss
                if training:
                    loss.backward()
                    self.optimizer.step()

            batch_size = x.size(0)
            total_loss += loss.item() * batch_size
            fruit_correct += (fruit_logits.argmax(dim=1) == fruit_target).sum().item()
            fresh_correct += (fresh_logits.argmax(dim=1) == fresh_target).sum().item()
            sample_count += batch_size

        return (
            total_loss / sample_count,
            fruit_correct / sample_count,
            fresh_correct / sample_count,
        )

    def fit_model(self, train_loader, val_loader, epochs=5):
        for epoch in range(epochs):
            train_metrics = self._run_epoch(train_loader, training=True)
            val_metrics = self._run_epoch(val_loader, training=False)

            for key, value in zip(
                ('train_loss', 'train_acc_fruit', 'train_acc_fresh'), train_metrics
            ):
                self.history[key].append(value)
            for key, value in zip(
                ('val_loss', 'val_acc_fruit', 'val_acc_fresh'), val_metrics
            ):
                self.history[key].append(value)

            print(
                f'[Epoch: {epoch}] '
                f'Train: [loss: {train_metrics[0]:.3f}, fruit: {train_metrics[1]:.3f}, fresh: {train_metrics[2]:.3f}] '
                f'Val: [loss: {val_metrics[0]:.3f}, fruit: {val_metrics[1]:.3f}, fresh: {val_metrics[2]:.3f}]'
            )

    def describe(self):
        print(self)
        print('\nModel structure:')
        print(self.base)
        print('\nBlock1:')
        print(self.block1)
        print('\nBlock2:')
        print(self.block2)
        print('\nBlock3:')
        print(self.block3)


In [ ]:

model_vgg16 = ModelVGG16().to(device)

In [ ]:
from torchinfo import summary
print(summary(model_vgg16, input_size=(BATCH_SIZE, 3, 244, 244)))

# Training the model

In [ ]:

model_vgg16.fit_model(train_loader, val_loader, epochs=5)

In [ ]:
model_vgg16.describe()

# Plotting Model Results

In [ ]:
plt.figure(figsize = (16, 4))

plt.subplot(1,3,1)
plt.title('Loss')
plt.plot(model_vgg16.history['train_loss'], label = 'Training')
plt.plot(model_vgg16.history['val_loss'], label = 'Validation')
plt.legend()

plt.subplot(1,3,2)
plt.title('Fruit Name Accuracy')
plt.plot(model_vgg16.history['train_acc_fruit'], label = 'Training')
plt.plot(model_vgg16.history['val_acc_fruit'], label = 'Training')
plt.legend()

plt.subplot(1,3,3)
plt.title('Freshness Accuracy')
plt.plot(model_vgg16.history['train_acc_fresh'], label = 'Training')
plt.plot(model_vgg16.history['val_acc_fresh'], label = 'Training')
plt.legend()


# Model Predictions

In [ ]:
preds1, preds2, fruit, fresh = [], [], [], []
model_vgg16.eval()

with torch.no_grad():
    for x, y1, y2 in tqdm(test_loader):
        pred = model_vgg16(x.to(device))
        
        pred1 = torch.argmax(pred[0], axis = 1).detach().cpu().numpy()
        pred2 = torch.argmax(pred[1], axis = 1).detach().cpu().numpy()
        preds1.extend(pred1)
        preds2.extend(pred2)
        fruit.extend(y1)
        fresh.extend(y2)
        
        
len(fruit), len(fresh), len(preds1), len(preds2)

In [ ]:
from sklearn.metrics import confusion_matrix

class_names = fruit_encoder.inverse_transform(np.arange(0, 9))

cm = confusion_matrix(fruit, preds1)
sea.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True, xticklabels = class_names, yticklabels = class_names)

plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix for Fruit Names')

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(fresh, preds2)
sea.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True, xticklabels = ['Fresh', 'Spoiled'], yticklabels = ['Fresh', 'Spoiled'])

plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix for Freshness')

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(fruit, preds1, target_names = class_names))
print(classification_report(fresh, preds2, target_names = ['Fresh', 'Spoiled']))